# 05_train_lgbm — LightGBM Ranker/Classifierの学習と評価(季節性特徴量版、結果的に不採用)

`04_feature_engineering.ipynb` で作った候補×特徴量テーブル(6候補ソース、学習4週+検証1週、
季節性特徴量[i_sales_yoy_7d/cat_sales_yoy_7d/cat_sales_recent_28d/cat_seasonal_lift]を含む)で
LightGBMを学習し、LambdaRank(LGBMRanker)とbinary分類(LGBMClassifier)を比較する。
読み込み・ダウンキャスト・負例サンプリングは`training_data.py`に共通化済み(OOM対策込み、
00_md/pipeline_build_log.md 6節参照)のものを使う。

評価はvalセット(cutoff = 最終日-7日)に対する**真の実購入集合**(候補プールに入っていない購入も含む)でのMAP@12。

**結果メモ(2026-09-18)**: この季節性特徴量版はmulti_week_eval.pyでの3週held-out検証では
0906版を一貫して上回った(平均0.026206 vs 0.026098)が、実際に提出するとPrivate LBが
0906版(0.02620、これまでの最高値)より明確に悪化した(0.02605、Public LBはわずかに改善)。
3週の多週検証でも検出できなかったケースであり、`06_predict_submit.ipynb`は0906版に復帰済み。
詳細は`00_md/pipeline_build_log.md`「季節性特徴量の追加」節参照。

In [ ]:
import os
import sys
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import lightgbm as lgb

sys.path.append(os.path.abspath("."))
from common import backtest_split, get_con, mapk, append_experiment_log, MODELS_DIR
from training_data import load_train_val, group_sizes, latest_feature_paths

train_path, val_path = latest_feature_paths()
print(f"train: {train_path}")
print(f"val:   {val_path}")

CAT_COLS = ["product_group_name", "index_group_name", "section_name"]

## 読み込み+ダウンキャスト+負例サンプリング(OOM対策、training_data.pyに共通化)

`_load_one`でpyarrow Table段階のうちにint8/int32/float32へcastしてからpandasに変換し
(生のint64/float64のまま全列展開されるピークを避ける)、trainを読み込み→ダウンキャスト→
負例サンプリングまで完全に終わらせてから、val用の生DataFrameを読み込む(2つの巨大な生DataFrameが
同時に存在しないようにする)。詳細はtraining_data.pyのdocstringと00_md/pipeline_build_log.md 6節参照。

負例サンプリング(陽性率0.08%と極端に不均衡なため):
- 正解を1件でも含むグループの行は**全て残す**(ランキング学習に必要な文脈を壊さないため、グループ単位で残す/落とすを決める。行単位で間引くとgroup引数と対応が崩れる)
- 正解を1件も含まないグループ(候補プールが実購入を1つも捉えられなかった顧客)は、グループ単位でランダムに`neg_group_sample_frac`の割合だけ残す
- valセットは評価用(early stoppingの参照値・最終MAP@12算出)なのでそのまま使う

In [ ]:
train_df, val_df, feature_cols = load_train_val(neg_group_sample_frac=0.15, random_state=42, verbose=True)

## 特徴量の確定

- カテゴリ列はvalの側をtrainのカテゴリ集合に合わせる(pyarrowはファイルごとに辞書順が独立なため)
- customer_idx/article_id/label/cutoffは特徴量から除外(product_codeは読み込み時点で除外済み)

In [ ]:
cat_cols = CAT_COLS

X_train, y_train = train_df[feature_cols], train_df["label"]
X_val, y_val = val_df[feature_cols], val_df["label"]

train_group_sizes = group_sizes(train_df, ["cutoff", "customer_idx"])
val_group_sizes = group_sizes(val_df, ["customer_idx"])

assert train_group_sizes.sum() == len(train_df)
assert val_group_sizes.sum() == len(val_df)
print(f"feature_cols ({len(feature_cols)}): {feature_cols}")
print(f"train groups: {len(train_group_sizes):,}, val groups: {len(val_group_sizes):,}")

## LGBMRanker(LambdaRank)

In [ ]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    n_estimators=300,
    num_leaves=63,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1,
)
ranker.fit(
    X_train, y_train,
    group=train_group_sizes,
    eval_set=[(X_val, y_val)],
    eval_group=[val_group_sizes],
    eval_at=[12],
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

## LGBMClassifier(binary)

正例率が約0.08〜0.09%と極端に不均衡なため `is_unbalance=True` を指定する。

In [ ]:
classifier = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    num_leaves=63,
    learning_rate=0.05,
    is_unbalance=True,
    random_state=42,
    verbosity=-1,
)
classifier.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="binary_logloss",
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

## 評価: valセットの真の実購入集合に対するMAP@12

候補プールに入っていない実購入も含めた完全な正解集合(`common.backtest_split`、02_baseline/03と同じ定義)で評価する。
候補プール自体のRecall上限(03の5ソース単純結合で0.088)を超えることはできない。

In [ ]:
con = get_con()
cutoff_check, actual_map = backtest_split(con, holdout_days=7)
print(f"cutoff_check={cutoff_check} (val_dfのcutoffと一致するはず: {val_df['cutoff'].iloc[0]})")
print(f"実購入があった顧客数: {len(actual_map):,}")

In [ ]:
def build_pred_map_from_scores(df, scores, k=12):
    tmp = df[["customer_idx", "article_id"]].copy()
    tmp["score"] = scores
    top = tmp.sort_values("score", ascending=False).groupby("customer_idx").head(k)
    return top.groupby("customer_idx")["article_id"].apply(list).to_dict()


val_df["score_ranker"] = ranker.predict(X_val)
val_df["score_classifier"] = classifier.predict_proba(X_val)[:, 1]

pred_map_ranker = build_pred_map_from_scores(val_df, val_df["score_ranker"])
pred_map_classifier = build_pred_map_from_scores(val_df, val_df["score_classifier"])

map_ranker = mapk(actual_map, pred_map_ranker, k=12)
map_classifier = mapk(actual_map, pred_map_classifier, k=12)

results_df = pd.DataFrame([
    {"model": "LGBMRanker(LambdaRank, 5ソース×学習4週)", "map@12": map_ranker},
    {"model": "LGBMClassifier(binary, 5ソース×学習4週)", "map@12": map_classifier},
    {"model": "参考: 前回05(2週学習・4ソース)LGBMRanker", "map@12": 0.0322},
    {"model": "参考: 02_baselineハイブリッド", "map@12": 0.0248},
])
results_df

## 特徴量重要度

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "ranker_gain": ranker.booster_.feature_importance(importance_type="gain"),
    "classifier_gain": classifier.booster_.feature_importance(importance_type="gain"),
}).sort_values("ranker_gain", ascending=False)
importance_df

## モデル保存と実験ログ記録

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
best_name = "ranker" if map_ranker >= map_classifier else "classifier"
best_model = ranker if best_name == "ranker" else classifier
best_score = max(map_ranker, map_classifier)

ranker_path = os.path.join(MODELS_DIR, f"model_{timestamp}_ranker.pkl")
classifier_path = os.path.join(MODELS_DIR, f"model_{timestamp}_classifier.pkl")
with open(ranker_path, "wb") as f:
    pickle.dump({"model": ranker, "feature_cols": feature_cols, "cat_cols": cat_cols}, f)
with open(classifier_path, "wb") as f:
    pickle.dump({"model": classifier, "feature_cols": feature_cols, "cat_cols": cat_cols}, f)
print(f"保存しました: {ranker_path}")
print(f"保存しました: {classifier_path}")

params_desc = f"n_estimators=300(early_stopping=30), num_leaves=63, learning_rate=0.05, neg_group_sample_frac={NEG_GROUP_SAMPLE_FRAC}"
features_desc = ", ".join(feature_cols)

append_experiment_log(
    run_id=f"run_{timestamp}_ranker", timestamp=timestamp, notebook="05_train_lgbm.ipynb",
    description="LGBMRanker(LambdaRank, group=customer_idx x cutoff)。04の候補×特徴量テーブル(5ソース×学習4週)で学習",
    features=features_desc, params=f"objective=lambdarank, {params_desc}",
    cv_score=round(map_ranker, 4), public_lb="", submission_file="", model_file=os.path.basename(ranker_path),
    notes="valは5候補ソース結合プール由来(学習は4週分)。評価は候補プール外の実購入も含む完全な正解集合に対するMAP@12",
)
append_experiment_log(
    run_id=f"run_{timestamp}_classifier", timestamp=timestamp, notebook="05_train_lgbm.ipynb",
    description="LGBMClassifier(binary, is_unbalance=True)。04の候補×特徴量テーブル(5ソース×学習4週)で学習",
    features=features_desc, params=f"objective=binary, is_unbalance=True, {params_desc}",
    cv_score=round(map_classifier, 4), public_lb="", submission_file="", model_file=os.path.basename(classifier_path),
    notes="valは5候補ソース結合プール由来(学習は4週分)。評価は候補プール外の実購入も含む完全な正解集合に対するMAP@12",
).tail(2)

## 次のステップ

- MAP@12が02_baselineを上回っていれば、cutoff=最終日(全データ)で候補生成→特徴量→予測を行い、
  全顧客(コールドスタートは人気度フォールバック)向けの提出ファイルを作って06として提出する
- 下回っていれば、特徴量重要度を見て弱いソース/特徴量を見直す、学習週を増やす、ハイパーパラメータを調整するなどの改善を検討する